# 03 · Join Sofascore + Capology — England Premier League 24/25

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2024/25 de Premier League inglesa**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_england_2425.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_england_2425.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  562 jugadores | 116 columnas
Capology:   770 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   brighton hove albion
   leicester city
   newcastle united
   tottenham hotspur
   west ham united

En Capology pero no en Sofascore:
   brighton
   leicester
   newcastle
   tottenham
   west ham


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [7]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'brighton':'brighton hove albion',
            'leicester':'leicester city',
            'newcastle':'newcastle united',
            'tottenham':'tottenham hotspur',
            'west ham':'west ham united'
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [8]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 516/562 (91.8%)
Sin emparejar: 46


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [9]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          6
Revisión media    (0.75 ≤ score < 0.90):   8
Revisión estricta (0.50 ≤ score < 0.75):   18
Revisión muy est. (score < 0.50):           14


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [10]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
1,Łukasz Fabiański,West Ham United,lukasz fabianski,0.968
11,Sam Morsy,Ipswich Town,samy morsy,0.947
9,Yehor Yarmolyuk,Brentford,yegor yarmolyuk,0.933
23,Mykhaylo Mudryk,Chelsea,mykhailo mudryk,0.933
36,Albert Grønbæk,Southampton,albert grnbaek,0.923
13,Joshua King,Fulham,josh king,0.900


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [11]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
5,Valentino Livramento,Newcastle United,tino livramento,0.857
0,Savinho,Manchester City,savio,0.833
27,Ben Brereton Díaz,Southampton,ben brereton,0.828
14,Edward Nketiah,Crystal Palace,eddie nketiah,0.815
33,Yunus Konak,Brentford,yunus emre konak,0.815
6,Stefan Ortega,Manchester City,stefan ortega moreno,0.788
20,Edmond-Paris Maghoma,Brentford,paris maghoma,0.788
8,Pape Matar Sarr,Tottenham Hotspur,pape sarr,0.750


In [12]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 8 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [13]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
12,Mads Roerslev,Brentford,mads roerslev rasmussen,0.722
25,Chido Obi-Martin,Manchester United,chido obi,0.720
24,Daniel Podence,Wolverhampton,daniel bentley,0.714
28,Bobby Decordova-Reid,Leicester City,bobby reid,0.667
38,Mateus Mané,Wolverhampton,matheus cunha,0.667
10,Emerson Palmieri,West Ham United,emerson,0.609
30,Kim Ji-Soo,Brentford,ji soo kim,0.600
43,George Edmundson,Ipswich Town,george hirst,0.571
32,Jake Evans,Leicester City,jamie vardy,0.571
21,Igor Júlio,Brighton & Hove Albion,igor,0.571


In [16]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['mads roerslev',
                    'chido obi martin',
                    'bobby decordova reid',
                    'emerson palmieri',
                    'kim ji soo',
                    'igor julio',
                    'andre',
                    'abdul fatawu issahaku'
                    


]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 8


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [17]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
15,Oliver Scarles,West Ham United,vladimir coufal,0.483
42,Remy Rees-Dottin,Bournemouth,ryan christie,0.483
17,Ben Winterburn,Bournemouth,luis sinisterra,0.483
16,Harry Howell,Brighton & Hove Albion,danny welbeck,0.480
44,Alfie Pond,Wolverhampton,luke cundle,0.476
18,Nathan Butler-Oyedeji,Arsenal,ethan nwaneri,0.471
40,Zain Silcott-Duberry,Bournemouth,alex scott,0.467
39,Jay Stansfield,Fulham,joachim andersen,0.467
34,Scott McTominay,Manchester United,mason mount,0.462
31,Divin Mubama,Manchester City,christian mcfarlane,0.452


In [18]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [19]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 538/562 (95.7%)
Sin salario:     24


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [20]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 24


,player,team,minutesPlayed,appearances,goals,assists
0,Nathan Butler-Oyedeji,Arsenal,11,1,0,0
1,Ben Winterburn,Bournemouth,22,4,0,0
2,Zain Silcott-Duberry,Bournemouth,1,1,0,0
3,Remy Rees-Dottin,Bournemouth,1,1,0,0
4,Billy Gilmour,Brighton & Hove Albion,97,2,0,0
5,Harry Howell,Brighton & Hove Albion,14,1,0,0
6,Mathis Amougou,Chelsea,12,1,0,0
7,Shumaira Mheuka,Chelsea,1,1,0,0
8,Roman Dixon,Everton,90,1,0,0
9,Jay Stansfield,Fulham,1,1,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [21]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Arsenal  —  SF sin salario:


,player,minutesPlayed
0,Nathan Butler-Oyedeji,11


  CG plantilla completa:


,player,player_norm
0,Albert Sambi Lokonga,albert sambi lokonga
1,Ben White,ben white
2,Bukayo Saka,bukayo saka
3,David Raya,david raya
4,Declan Rice,declan rice
5,Ethan Nwaneri,ethan nwaneri
6,Fábio Vieira,fabio vieira
7,Gabriel Jesus,gabriel jesus
8,Gabriel Magalhães,gabriel magalhaes
9,Gabriel Martinelli,gabriel martinelli



  Bournemouth  —  SF sin salario:


...

,player,player_norm
0,Aaron Cresswell,aaron cresswell
1,Aaron Wan-Bissaka,aaron wan bissaka
2,Alphonse Areola,alphonse areola
3,Andy Irving,andy irving
4,Callum Marshall,callum marshall
5,Carlos Soler,carlos soler
6,Crysencio Summerville,crysencio summerville
7,Danny Ings,danny ings
8,Edson Álvarez,edson alvarez
9,Emerson,emerson



  Wolverhampton  —  SF sin salario:


,player,minutesPlayed
0,Alfie Pond,1
1,Daniel Podence,46
2,Mateus Mané,1


  CG plantilla completa:


,player,player_norm
0,André Trindade,andre trindade
1,Bastien Meupiyou,bastien meupiyou
2,Boubacar Traoré,boubacar traore
3,Carlos Forbs,carlos forbs
4,Chem Campbell,chem campbell
5,Chiquinho,chiquinho
6,Craig Dawson,craig dawson
7,Daniel Bentley,daniel bentley
8,Dexter Lembikisa,dexter lembikisa
9,Emmanuel Agbadou,emmanuel agbadou


In [22]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {

}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 0


In [23]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')


Tras matches manuales: 538/562 (95.7%)


In [25]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [24]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_england_2425.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_england_2425.csv
   Jugadores totales:  562
   Con salario:        538
   Sin salario (NaN):  24
   Columnas:           121
